# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

# Model Finetuning consists of the following steps

1. Loading the data, with huggingface's `datasets` library.
   1. Transform the dataset to fit the finetuning data format.
2. Load the model with huggingface's `transformer` library.
   1. Optionally, setup `bitsandbytes` config for quantized
   2. Depending on hardware (GPU), you'll need to decide the float point math of loading (bf16 vs fp16)
3. Setup trainer
   1. Setup LoRA training with `peft`, optionally setup quantization for QLoRA.
   2. With `trl` SFTConfig, SFTTrainer.
   3. Setup Optimizer.
4. Setup wandb or other training tracking.
5. TRAIN!!!
   1. Setup auto-saving
   2. Load from previous save point.
6. Merge the LoRA trained layers and save it.



In [ ]:
%uv pip install transformers datasets huggingface_hub accelerate bitsandbytes wandb peft trl dotenv

Using Python 3.12.6 environment at: /usr/local
Audited 9 packages in 21ms
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

CACHE_DIR = "/mnt/huggingface-cache"
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

# set huggingface cache path
os.environ['HF_HOME'] = CACHE_DIR


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])
wandb.init(
    entity='hxy9243-personal',
    project='test-project'
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hxy9243 (hxy9243-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.22.3
wandb: Run data is saved locally in /root/wandb/run-20251104_083231-stqiep0u
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run absurd-dragon-7
wandb: ⭐️ View project at https://wandb.ai/hxy9243-personal/test-project
wandb: 🚀 View run at https://wandb.ai/hxy9243-personal/test-project/runs/stqiep0u


In [ ]:
from pprint import pprint

from datasets import load_dataset

dataset_name = "iamtarun/python_code_instructions_18k_alpaca"
dataset = load_dataset(dataset_name, split='train')

pprint(dataset[0])
print(len(dataset))

def create_prompt_and_completion(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n"
    if example.get("input"):
        prompt += f"### Input:\n{example['input']}\n\n"

    completion = example['output']
    return {"prompt": prompt, "completion": completion}

formatted_dataset = dataset.map(
    create_prompt_and_completion,
    remove_columns=dataset.column_names
)

{'input': '[1, 2, 3, 4, 5]',
 'instruction': 'Create a function to calculate the sum of a sequence of '
                'integers.',
 'output': '# Python code\n'
           'def sum_sequence(sequence):\n'
           '  sum = 0\n'
           '  for num in sequence:\n'
           '    sum += num\n'
           '  return sum',
 'prompt': 'Below is an instruction that describes a task. Write a response '
           'that appropriately completes the request.\n'
           '\n'
           '### Instruction:\n'
           'Create a function to calculate the sum of a sequence of integers.\n'
           '\n'
           '### Input:\n'
           '[1, 2, 3, 4, 5]\n'
           '\n'
           '### Output:\n'
           '# Python code\n'
           'def sum_sequence(sequence):\n'
           '  sum = 0\n'
           '  for num in sequence:\n'
           '    sum += num\n'
           '  return sum'}
18612


In [ ]:
# Import necessary libraries for fine-tuning
import torch
import wandb
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

NEW_MODEL = "phi-3-mini-4k-instruct-python-code"

# LoRA configuration
lora_r = 16
lora_alpha = 32
lora_dropout = 0.05

# LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

# Setup training parameters
training_arguments = SFTConfig(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    save_strategy="steps",    # Save checkpoints at regular intervals
    save_steps=500,           # Save every 500 steps
    save_total_limit=3,       # Keep only the 3 most recent checkpoints
    logging_steps=5,
    learning_rate=2e-4,
    weight_decay=0.001,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="wandb",
    dataset_text_field='prompt',
)
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    args=training_arguments,
)


`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [ ]:
new_model = "phi3-finetuned-python"

def run():
    trainer.train(resume_from_checkpoint=True)
    trainer.model.save_pretrained(new_model)

run()

Step,Training Loss


In [ ]:
from peft import LoraConfig, PeftModel

print("Merging LoRA adapters...")
model = trainer.model.merge_and_unload()
print("Merge complete.")

# Define where to save the merged model
merged_model_path = "/mnt/huggingface-cache/trained/phi-3-mini-4k-instruct-python-code-merged"

# Save the merged model
print(f"Saving merged model to: {merged_model_path}")
model.save_pretrained(merged_model_path)

Merging LoRA adapters...
Merge complete.
Saving merged model to: /mnt/huggingface-cache/trained/phi-3-mini-4k-instruct-python-code-merged


In [ ]:
import torch

import gc
gc.collect()

# del model
# del trainer
torch.cuda.
torch.cuda.empty_cache()